# Computer Vision HW4
Reyhaneh Hajebi  _   810100116

Safora Alavipanah  _  810100254


## Introduction

Image classification is one of the fundamental tasks in computer vision and has numerous applications such as scene recognition, autonomous driving, medical imaging, and intelligent surveillance systems. In recent years, Convolutional Neural Networks (CNNs) have become the dominant approach for image classification due to their ability to automatically extract hierarchical features from raw pixel data. Unlike traditional machine learning methods that require manual feature engineering, CNNs learn meaningful representations directly from images through multiple convolutional and pooling layers.

In this project, a multi-class image classification model is designed and evaluated using the Intel Image Classification dataset, which consists of six natural scene categories: buildings, forest, glacier, mountain, sea, and street. 

The objective is to implement a complete deep learning pipeline including data loading, preprocessing, data augmentation, model design, training, evaluation, and performance analysis. 

In addition, an improved version of the model is developed by incorporating techniques such as Batch Normalization and Dropout to reduce overfitting and improve generalization performance. 

The experimental results, including accuracy, loss curves, confusion matrix, and classification report, are analyzed to evaluate the effectiveness of the proposed models and to gain a deeper understanding of CNN-based image classification.

## 1. Data Loading
- Print batch shape
- Validation split
- Show samples
- Explain image size and flatten.

In [ ]:
import tensorflow as tf, matplotlib.pyplot as plt

train_dir='seg_train'
test_dir='seg_test'
img_size=(150,150);batch_size=32
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)
test_ds=tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=img_size,
    batch_size=batch_size
)
for x,y in train_ds.take(1):
    print(x.shape,y.shape)
class_names=train_ds.class_names
plt.figure(figsize=(8,8))
for imgs,labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3,3,i+1);plt.imshow(imgs[i].numpy().astype('uint8'));plt.title(class_names[labels[i]]);plt.axis('off')


Images are 150x150 RGB. 

**Why Flatten?** Flatten converts 3D feature maps to 1D before Dense layers. This is required because fully connected layers in neural networks expect 1D input. Flattening preserves all pixel information while enabling the model to learn global patterns, though it loses spatial structure

## 2. Preprocessing

**tf.keras.layers.RandomFlip("horizontal")**

- Randomly flips images left-to-right (mirror image) with a 50% probability per image.

- Useful for objects that are symmetric (e.g., faces, cars, general objects) to improve generalization without changing semantics.

**tf.keras.layers.RandomRotation(0.1)**

- Randomly rotates images by up to 10% of a full circle (i.e., ±36 degrees, since 0.1 × 360° = 36°).

- Values can be in radians (e.g., 0.1 radians ≈ 5.7°) or as a fraction of 2π when specified (here it's typically fraction of 2π in Keras). This helps models become invariant to small orientation changes.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1)
])

# Normalization
normalization_layer = tf.keras.layers.Rescaling(1./255)

Normalization scales pixels from [0,255] to [0,1]. 

**Augmentation reduces overfitting by increasing diversity.**

Data augmentation artificially increases the diversity of the training dataset by generating modified versions of existing images through random transformations such as flipping and rotation. This exposes the model to more variations during training, reducing its tendency to memorize the training data and improving its ability to generalize to unseen images. As a result, data augmentation helps reduce overfitting.

## 3. CNN Model

In [ ]:
model=tf.keras.Sequential([
tf.keras.Input(shape=(150,150,3)),
data_augmentation,
tf.keras.layers.Rescaling(1./255),
tf.keras.layers.Conv2D(32,3,activation='relu'),tf.keras.layers.MaxPooling2D(),
tf.keras.layers.Conv2D(64,3,activation='relu'),tf.keras.layers.MaxPooling2D(),
tf.keras.layers.Conv2D(128,3,activation='relu'),tf.keras.layers.MaxPooling2D(),
tf.keras.layers.Conv2D(256,3,activation='relu'),tf.keras.layers.MaxPooling2D(),
tf.keras.layers.Flatten(),
tf.keras.layers.Dense(128,activation='relu'),
tf.keras.layers.Dense(6,activation='softmax')
])
model.summary()

Conv layers extract features; pooling reduces size; flatten prepares dense input.

The proposed CNN model consists of four convolutional layers followed by four max-pooling layers, a flatten layer, one fully connected (Dense) hidden layer, and a final output layer with six neurons corresponding to the six classes of the dataset. The convolutional layers use 32, 64, 128, and 256 filters, respectively, to gradually learn low-level features such as edges and textures and higher-level semantic features such as object shapes and scene structures. The MaxPooling layers reduce the spatial dimensions of the feature maps, decreasing computational complexity while preserving the most important information. The Flatten layer converts the extracted two-dimensional feature maps into a one-dimensional feature vector that can be processed by the Dense layers for classification.

The model contains approximately **1.99 million trainable parameters**, with the majority of them located in the first Dense layer (1,605,760 parameters). This large number of parameters is due to connecting the flattened feature vector of size 12,544 to 128 neurons, making the Dense layer responsible for learning complex decision boundaries between the six classes. In contrast, the convolutional layers contain significantly fewer parameters because they use local receptive fields and shared weights, making feature extraction computationally efficient. Finally, the Softmax output layer produces a probability distribution over the six scene categories, allowing the model to predict the most likely class for each input image.


## 4. Loss and Optimizer

In [ ]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

Sparse categorical crossentropy suits integer labels for multiclass classification.

## 5. Training

In [ ]:
history=model.fit(train_ds,validation_data=val_ds,epochs=10)

## 6. Evaluation

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix
import numpy as np
loss,acc=model.evaluate(test_ds)
print(f"loss = {loss},\naccuracy = {acc}")
y_true=[];y_pred=[]
for x,y in test_ds:
 p=np.argmax(model.predict(x,verbose=0),1)
 y_true.extend(y.numpy());y_pred.extend(p)
print("Classification Report: \n",classification_report(y_true,y_pred,target_names=class_names))
print("Confusion Matrix: \n", confusion_matrix(y_true,y_pred))

**Brief explanation for choosing Crossentropy**

Sparse Categorical Crossentropy was selected as the loss function because the problem involves **multi-class classification**, where each image belongs to exactly one of six classes. This loss function measures the difference between the predicted probability distribution produced by the Softmax output layer and the true class label. Since the dataset labels are represented as integer values rather than one-hot encoded vectors, the sparse version is more efficient and avoids unnecessary memory usage while providing the same optimization objective. Combined with the Softmax activation function, it enables the model to learn accurate class probabilities during training.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 12))

for images, labels in test_ds.take(1):
    predictions = model.predict(images, verbose=0)
    pred_labels = np.argmax(predictions, axis=1)

    for i in range(9):
        plt.subplot(3, 3, i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(
            f"True: {class_names[labels[i]]}\nPred: {class_names[pred_labels[i]]}",
            fontsize=9
        )
        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

wrong_images = []
wrong_true = []
wrong_pred = []

for images, labels in test_ds:
    preds = np.argmax(model.predict(images, verbose=0), axis=1)

    for img, true, pred in zip(images, labels.numpy(), preds):
        if true != pred:
            wrong_images.append(img.numpy().astype("uint8"))
            wrong_true.append(true)
            wrong_pred.append(pred)

        if len(wrong_images) >= 9:
            break
    if len(wrong_images) >= 9:
        break

plt.figure(figsize=(12,12))

for i in range(len(wrong_images)):
    plt.subplot(3,3,i+1)
    plt.imshow(wrong_images[i])
    plt.title(
        f"True: {class_names[wrong_true[i]]}\nPred: {class_names[wrong_pred[i]]}",
        fontsize=9
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

The misclassified samples indicate that the model tends to confuse scene categories that share similar visual characteristics. For example, images of **mountains and glaciers** may be misclassified because both contain rocky landscapes and snowy textures, while **buildings and streets** can appear similar due to the presence of urban structures. Likewise, **sea and glacier** images may occasionally be confused because both contain large blue and white regions. These errors suggest that although the CNN successfully learns discriminative features, some classes exhibit overlapping visual patterns that make classification challenging. The confusion matrix provides further insight into which class pairs are most frequently confused and can guide future improvements such as increasing model capacity, applying stronger data augmentation, or using transfer learning.


## 7. Curves

In [ ]:
import matplotlib.pyplot as plt

# Accuracy Plot
plt.figure(figsize=(7,5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Loss Plot
plt.figure(figsize=(7,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

The training and validation accuracy curves show how the model's classification performance improves over successive epochs. As training progresses, the training accuracy increases while the validation accuracy also improves and eventually stabilizes, indicating that the model is learning meaningful features from the dataset. Similarly, the training and validation loss decrease throughout the training process, showing that the prediction error becomes smaller as the model parameters are optimized. The gap between the training and validation curves provides insight into the model's ability to generalize to unseen data.

 The model generalizes well. Stopping around epoch 6 or 7 would avoid the minor overfitting seen at epoch 8. No underfitting is present.

In general, the plot shows a well-fitted model with no significant overfitting or underfitting.

- **No underfitting:** Both training and validation accuracy increase steadily from the start, while both losses decrease consistently. This means the model is learning effectively.

- **No significant overfitting:** Training and validation metrics remain very close throughout. There is no large gap where training accuracy is much higher than validation accuracy, nor a rise in validation loss.

- **Slight late-phase balance:** Toward the end, training accuracy slightly exceeds validation accuracy, and training loss falls a bit below validation loss. This suggests minimal overfitting starting, but it is not severe enough to harm generalization.



## 8. Improved Model

This model uses three improvements:

Batch Normalization
Dropout
Smaller learning rate (0.0005)

and generally performs better than the baseline CNN.

In [ ]:
from tensorflow.keras import layers, models

improved = models.Sequential([

    tf.keras.Input(shape=(150,150,3)),

    data_augmentation,
    layers.Rescaling(1./255),

    # Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Block 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    # Block 4
    layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),

    layers.Dropout(0.2),

    layers.Dense(6, activation='softmax')
])

improved.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = improved.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)


### Evaluate on the test set:

In [ ]:
test_loss, test_acc = improved.evaluate(test_ds)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

### Plot training curves

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.figure(figsize=(7,5))
plt.plot(history2.history['accuracy'], label='Training Accuracy')
plt.plot(history2.history['val_accuracy'], label='Validation Accuracy')
plt.title('Improved Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

# Loss
plt.figure(figsize=(7,5))
plt.plot(history2.history['loss'], label='Training Loss')
plt.plot(history2.history['val_loss'], label='Validation Loss')
plt.title('Improved Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

To improve the performance of the baseline CNN model, three modifications were introduced: 
- Batch Normalization layers were added after each convolutional layer
- Dropout layer with a rate of 0.5 was inserted before the final classification layer
- the Adam optimizer learning rate was reduced from 0.001 to 0.0005

Batch Normalization stabilizes the distribution of intermediate feature maps and accelerates convergence, while Dropout randomly deactivates neurons during training to reduce overfitting. Reducing the learning rate allows the optimizer to make smaller and more stable parameter updates, leading to better convergence.

Compared with the original model, the improved model achieved higher validation and test accuracy while producing smoother learning curves. The gap between the training and validation accuracy became smaller, indicating improved generalization ability and reduced overfitting. In addition, the validation loss decreased more consistently throughout training, demonstrating that the model learned more robust image representations. Overall, these modifications resulted in a more stable and accurate classifier for the Intel Image Classification dataset.
